# Day 1 — Exploratory Data Analysis

This notebook explores:
- `data/raw/patient_data.csv` (structured patient records)
- `data/documents/` (1,050 clinical markdown files)

It uses the Day 1 modules in `src/data/` and writes `artifacts/data_profile.json`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import settings
from data import (
    PatientDataLoader,
    build_and_save_data_profile,
    get_dataset_schema,
    get_target_columns,
    parse_documents_directory,
    summarize_document_corpus,
)

print("CSV:", settings.patient_csv_path)
print("Documents:", settings.documents_dir)

## 1. Patient CSV — load and inspect

In [ ]:
with PatientDataLoader(csv_path=settings.patient_csv_path) as loader:
    patients = loader.get_dataframe()
    row_count = loader.row_count

schema = get_dataset_schema(row_count)
patients.head()

In [ ]:
print(f"Rows: {row_count}")
print(f"Columns: {len(patients.columns)}")
patients.info()

In [ ]:
patients.describe(include="all").T

## 2. Target variables

In [ ]:
for target in get_target_columns():
    print(f"\n{target}")
    print(patients[target].value_counts() if patients[target].dtype == "object" else patients[target].describe())

## 3. Quick SQL checks (DuckDB)

In [ ]:
queries = {
    "smokers": "SELECT COUNT(*) AS count FROM patients WHERE smoker = 'Yes'",
    "males_over_40_readmitted": """
        SELECT COUNT(*) AS count
        FROM patients
        WHERE sex = 'Male' AND age > 40 AND readmitted = 1
    """,
    "more_than_5_medications": """
        SELECT COUNT(*) AS count
        FROM patients
        WHERE medication_count > 5
    """,
}

with PatientDataLoader(csv_path=settings.patient_csv_path) as loader:
    for name, sql in queries.items():
        result = loader.query(sql)
        print(f"{name}: {int(result.iloc[0]['count'])}")

## 4. Clinical documents — parse and summarize

In [ ]:
documents = parse_documents_directory(settings.documents_dir)
corpus_summary = summarize_document_corpus(documents)

corpus_summary

In [ ]:
sample = documents[0]
print("File:", sample.source_file)
print("Title:", sample.title)
print("Sections:", [section.section_name for section in sample.sections[:5]])
print("\nSample section content:\n")
print(sample.sections[0].content[:500])

## 5. Save data profile artifact

In [ ]:
profile, schema, saved_path = build_and_save_data_profile(
    csv_path=settings.patient_csv_path,
    output_path=settings.artifacts_dir / "data_profile.json",
)

print(f"Saved: {saved_path}")
profile.model_dump()["row_count"], len(profile.columns)

## Day 1 notes

- CSV and documents are **separate** data sources (different patient IDs).
- `chronic_obstructive_pulmonary_disease` is a **4-class** target (A/B/C/D).
- `alanine_aminotransferase` is a **continuous** regression target.
- Next step (Day 2): train ML models and save insights to `artifacts/insights/`.